In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sentence_transformers import SentenceTransformer
import faiss

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Vector store:", VECTOR_STORE_DIR)

c:\Users\shikh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform
Vector store: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\vector_store


In [2]:
faq_files = list(
    DATA_DIR.rglob("faq_knowledge_base_150.csv")
)

print("FAQ files found:")

for file in faq_files:
    print(file)

FAQ files found:
c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\knowledge_base\faq_knowledge_base_150.csv


In [3]:
if not faq_files:
    raise FileNotFoundError(
        "faq_knowledge_base_150.csv was not found inside data/"
    )

faq_path = faq_files[0]

faq_df = pd.read_csv(faq_path)

print("FAQ shape:", faq_df.shape)
print("\nColumns:")
print(faq_df.columns.tolist())

faq_df.head()

FAQ shape: (150, 4)

Columns:
['faq_id', 'question', 'answer', 'category']


,faq_id,question,answer,category
0,FAQ001,How do I track my order?,Go to My Orders > Select Order > Track Package...,Delivery Issue
1,FAQ002,What should I do if my order is delayed?,Check tracking first. If no update for 48 hour...,Delivery Issue
2,FAQ003,Can I change my delivery address after placing...,"Yes, before dispatch. Go to My Orders > Order ...",Delivery Issue
3,FAQ004,What happens if I miss my delivery?,Courier will reattempt delivery. You can also ...,Delivery Issue
4,FAQ005,How do I reschedule my delivery?,Use the tracking link in your SMS/email to sel...,Delivery Issue


In [4]:
print(faq_df.isnull().sum())

faq_id      0
question    0
answer      0
category    0
dtype: int64


In [5]:
print("\nCategories:")
print(faq_df["category"].value_counts())


Categories:
category
Product Issue               27
Account & Login             27
Payment Issue               24
Delivery Issue              22
Refund & Return             18
App & Website Issue         17
Seller & Product Listing    15
Name: count, dtype: int64


In [6]:
faq_df["question"] = faq_df["question"].fillna("").astype(str)
faq_df["answer"] = faq_df["answer"].fillna("").astype(str)
faq_df["category"] = faq_df["category"].fillna("Unknown").astype(str)

faq_df["question"] = (
    faq_df["question"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

faq_df["answer"] = (
    faq_df["answer"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

faq_df["category"] = (
    faq_df["category"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Cleaning completed.")

Cleaning completed.


In [7]:
documents = []

for _, row in faq_df.iterrows():

    text = f"""
Category: {row['category']}

Question: {row['question']}

Answer: {row['answer']}
""".strip()

    documents.append({
        "faq_id": row["faq_id"],
        "category": row["category"],
        "question": row["question"],
        "answer": row["answer"],
        "text": text
    })

print("Number of documents:", len(documents))

print("\nSample document:")
print(documents[0]["text"])

Number of documents: 150

Sample document:
Category: Delivery Issue

Question: How do I track my order?

Answer: Go to My Orders > Select Order > Track Package for real-time courier updates.


In [8]:
def chunk_text(
    text,
    chunk_size=500,
    chunk_overlap=100
):
    """
    Split text into overlapping character chunks.
    """

    if len(text) <= chunk_size:
        return [text]

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - chunk_overlap

    return chunks

In [9]:
chunks = []

for doc in documents:

    doc_chunks = chunk_text(
        doc["text"],
        chunk_size=500,
        chunk_overlap=100
    )

    for i, chunk in enumerate(doc_chunks):

        chunks.append({
            "chunk_id": f"{doc['faq_id']}_{i}",
            "faq_id": doc["faq_id"],
            "category": doc["category"],
            "question": doc["question"],
            "answer": doc["answer"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 150


In [10]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    embedding_model_name
)

print("Embedding model loaded successfully.")

Embedding model loaded successfully.


In [11]:
import sys

!{sys.executable} -m pip install --force-reinstall "numpy==1.26.4"

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.20.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import numpy
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 5/5 [00:01<00:00,  3.57it/s]

Embedding shape: (150, 384)


In [13]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(embeddings)

print("FAISS index created successfully.")
print("Number of vectors:", index.ntotal)

FAISS index created successfully.
Number of vectors: 150


In [14]:
def retrieve_documents(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx == -1:
            continue

        result = chunks[idx].copy()

        result["score"] = float(score)

        results.append(result)

    return results

In [15]:
query = "How can I return a damaged product?"

results = retrieve_documents(
    query,
    top_k=5
)

for i, result in enumerate(results, 1):

    print("=" * 70)

    print("Result:", i)
    print("Score:", round(result["score"], 4))
    print("Category:", result["category"])
    print("Question:", result["question"])
    print("Answer:", result["answer"])

Result: 1
Score: 0.6194
Category: Product Issue
Question: Can I get a product replaced instead of refunded?
Answer: Yes, replacement is available for defective and wrong items within the return window.
Result: 2
Score: 0.5269
Category: Refund & Return
Question: Can I return a product after the return window?
Answer: Generally no, but exceptions can be made for manufacturing defects. Contact support.
Result: 3
Score: 0.5171
Category: Delivery Issue
Question: What if my package is damaged on arrival?
Answer: Do not accept if visibly damaged. If opened and damaged, report via My Orders within 48 hours.
Result: 4
Score: 0.5069
Category: Product Issue
Question: What if product quality doesn't match description?
Answer: You can return for full refund or replacement within the return window.
Result: 5
Score: 0.5039
Category: Product Issue
Question: How do I report a defective product?
Answer: Go to My Orders > Report > Defective Item. Upload photos for faster resolution.


In [16]:
def retrieve_by_category(
    query,
    category,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        min(index.ntotal, 20)
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx == -1:
            continue

        result = chunks[idx]

        if (
            result["category"].lower().strip()
            ==
            category.lower().strip()
        ):

            result_copy = result.copy()

            result_copy["score"] = float(score)

            results.append(result_copy)

            if len(results) == top_k:
                break

    return results

In [17]:
results = retrieve_by_category(
    query="I want a refund for my damaged product",
    category="Refund",
    top_k=5
)

for result in results:

    print(
        result["score"],
        result["question"]
    )

In [18]:
index_path = (
    VECTOR_STORE_DIR /
    "faq_faiss.index"
)

faiss.write_index(
    index,
    str(index_path)
)

print(
    "FAISS index saved at:",
    index_path
)

FAISS index saved at: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\vector_store\faq_faiss.index


In [19]:
metadata_path = (
    VECTOR_STORE_DIR /
    "faq_metadata.pkl"
)

with open(
    metadata_path,
    "wb"
) as f:

    pickle.dump(
        chunks,
        f
    )

print(
    "Metadata saved at:",
    metadata_path
)

Metadata saved at: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\vector_store\faq_metadata.pkl


In [20]:
config = {
    "embedding_model": embedding_model_name,
    "dimension": dimension,
    "total_vectors": index.ntotal
}

config_path = (
    VECTOR_STORE_DIR /
    "rag_config.pkl"
)

with open(
    config_path,
    "wb"
) as f:

    pickle.dump(
        config,
        f
    )

print("RAG configuration saved.")

RAG configuration saved.


In [21]:
import sys

!{sys.executable} -m pip install --force-reinstall "numpy==1.26.4"

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.20.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import numpy as np
import torch

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("NumPy test:", np.array([1, 2, 3]))

NumPy: 1.26.4
PyTorch: 2.3.1+cpu
NumPy test: [1 2 3]
